# Stage 05 — Diagnostics

Run all 12 automated checks and write `data/results/diagnostics_flags.json`.
Follow `skills/stage_05.md` for detailed instructions.

**Checks:**
1. Replication pass
2. DML coefficient direction
3. DML confidence interval coverage
4. Nuisance model quality
5. Sample size
6. HTE heterogeneity signal
7. Causal forest ATE consistency (skip — no CF results)
8. Causal forest CATE plausibility (skip — no CF results)
9. ATE SE plausibility (skip — no CF results)
10. Cross-fitting stability
11. Learner sign agreement
12. Lasso variable selection

In [1]:
from paths import *
import json
import numpy as np

spec      = json.loads(PAPER_SPEC.read_text())
rep_check = json.loads(REPLICATION_CHECK.read_text())
rep_res   = json.loads(REPLICATION_RESULTS.read_text())
dml_res   = json.loads(DML_RESULTS.read_text())

# Load optional files
HTE_RESULTS = RESULTS_DIR / 'hte_results.json'
hte_res = json.loads(HTE_RESULTS.read_text()) if HTE_RESULTS.exists() else None

CF_RESULTS = RESULTS_DIR / 'causal_forest_results.json'
cf_res = json.loads(CF_RESULTS.read_text()) if CF_RESULTS.exists() else None

# ── Published reference values ──
pub_main = spec['main_results'][0]  # DML (Lasso) reference from B&N
pub_coef = pub_main['coef']         # -0.178
pub_se   = pub_main.get('se')
pub_n    = pub_main.get('n_obs')

# ── DML preferred (our replication) ──
dml_coef = dml_res['preferred_coef']
dml_se   = dml_res['preferred_se']
dml_lo   = dml_res['preferred_ci_lo']
dml_hi   = dml_res['preferred_ci_hi']

# ── Identify the primary spec: Investment2005 ~ effective_5yr ──
primary_spec = None
for s in dml_res['specifications']:
    out_var = s['outcome']['variable'] if isinstance(s['outcome'], dict) else s['outcome']
    trt_var = s['treatment']['variable'] if isinstance(s['treatment'], dict) else s['treatment']
    if out_var == 'Investment2005' and trt_var == 'effective_5yr':
        primary_spec = s
        break
assert primary_spec is not None, "Could not find primary spec Investment2005 ~ effective_5yr"

print(f'Published main coef (B&N DML Lasso): {pub_coef}')
print(f'Published SE: {pub_se}')
print(f'Our DML preferred coef ({dml_res["preferred_learner"]}): {dml_coef:.6f}  CI: [{dml_lo:.6f}, {dml_hi:.6f}]')
print(f'Primary spec best learner: {primary_spec["learners"]["Best"]["best_outcome_method"]}')
print()

# ── Summary of all 12 specs ──
print(f'Total DML specifications: {len(dml_res["specifications"])}')
for s in dml_res['specifications']:
    out_var = s['outcome']['variable'] if isinstance(s['outcome'], dict) else s['outcome']
    trt_var = s['treatment']['variable'] if isinstance(s['treatment'], dict) else s['treatment']
    best_entry = s['learners']['Best']
    print(f'  {out_var} ~ {trt_var}: best_coef={best_entry["coef"]:.4f}, p={best_entry["pval"]:.4f}, n={s["n_obs"]}')

print()
print(f'Replication check: pass={rep_check["pass"]}, worst_rel_diff={rep_check["worst_rel_diff_pct"]}%')
print(f'HTE results available: {hte_res is not None}')
print(f'Causal Forest results available: {cf_res is not None}')

Published main coef (B&N DML Lasso): -0.178
Published SE: 0.096
Our DML preferred coef (Best): -0.210963  CI: [-0.366322, -0.055604]
Primary spec best learner: Forest

Total DML specifications: 12
  Investment2005 ~ statutory: best_coef=-0.0509, p=0.5075, n=61
  Investment2005 ~ effective: best_coef=-0.1684, p=0.1784, n=61
  Investment2005 ~ effective_5yr: best_coef=-0.2110, p=0.0078, n=61
  FDI2005 ~ statutory: best_coef=-0.1433, p=0.0892, n=61
  FDI2005 ~ effective: best_coef=-0.1406, p=0.0872, n=61
  FDI2005 ~ effective_5yr: best_coef=-0.0370, p=0.7002, n=61
  nbusinessdensitycf ~ statutory: best_coef=-0.0893, p=0.1142, n=60
  nbusinessdensitycf ~ effective: best_coef=-0.1069, p=0.1140, n=60
  nbusinessdensitycf ~ effective_5yr: best_coef=-0.1216, p=0.0536, n=60
  nentryrateav ~ statutory: best_coef=-0.1443, p=0.0112, n=50
  nentryrateav ~ effective: best_coef=-0.1422, p=0.0161, n=50
  nentryrateav ~ effective_5yr: best_coef=-0.1726, p=0.0064, n=50

Replication check: pass=True, wor

In [2]:
def flag(value, pass_cond, warn_cond, note):
    """Return a diagnostic flag dict with status PASS/WARN/FAIL."""
    if pass_cond:
        return {'status': 'PASS', 'value': value, 'note': note}
    elif warn_cond:
        return {'status': 'WARN', 'value': value, 'note': note}
    else:
        return {'status': 'FAIL', 'value': value, 'note': note}

checks = {}

# ═══════════════════════════════════════════════════════════════
# Check 1: Replication pass
# PASS if worst_rel_diff_pct < 5, WARN if [5,15], FAIL if > 15
# ═══════════════════════════════════════════════════════════════
worst = rep_check['worst_rel_diff_pct']
n_compared = rep_res.get('n_compared', 0)
n_pass = rep_res.get('n_pass', 0)
n_total = rep_res.get('n_specs', 0)

# Check sign consistency across all specs with published reference
sign_matches = []
for s in rep_res['specs']:
    if s.get('published_coef') is not None and s['replicated_coef'] != 0:
        same = (s['published_coef'] * s['replicated_coef']) > 0 or (s['published_coef'] == 0 and s['replicated_coef'] == 0)
        sign_matches.append(same)
all_signs_correct = all(sign_matches) if sign_matches else True

note_parts = [f'{worst:.2f}% max deviation ({n_pass}/{n_compared} within threshold)']
if all_signs_correct:
    note_parts.append('all signs correct')
else:
    note_parts.append('SIGN MISMATCH in some specs')

checks['replication_pass'] = flag(
    worst,
    pass_cond=(worst < 5),
    warn_cond=(5 <= worst < 15),
    note='; '.join(note_parts)
)
print(f'Check 1 (Replication): {checks["replication_pass"]["status"]} -- {checks["replication_pass"]["note"]}')

# ═══════════════════════════════════════════════════════════════
# Check 2: DML coefficient direction
# PASS if same sign and relative diff < 50%
# WARN if same sign but rel diff >= 50%
# FAIL if sign change
# ═══════════════════════════════════════════════════════════════
same_sign = (pub_coef * dml_coef) > 0
if pub_coef != 0:
    rel_diff = abs(dml_coef - pub_coef) / abs(pub_coef)
else:
    rel_diff = float('inf') if dml_coef != 0 else 0.0

direction_note = f'our={dml_coef:.4f} vs published={pub_coef}; '
if same_sign:
    direction_note += f'same sign, rel_diff={rel_diff:.1%}'
else:
    direction_note += 'SIGN CHANGE'

checks['dml_direction'] = flag(
    round(dml_coef, 6),
    pass_cond=(same_sign and rel_diff < 0.5),
    warn_cond=(same_sign),
    note=direction_note
)
print(f'Check 2 (DML direction): {checks["dml_direction"]["status"]} -- {checks["dml_direction"]["note"]}')

# ═══════════════════════════════════════════════════════════════
# Check 3: DML confidence interval coverage
# PASS if published coef is inside DML CI
# WARN if outside but within 1 SE
# FAIL if > 2 SE outside
# ═══════════════════════════════════════════════════════════════
inside = dml_lo <= pub_coef <= dml_hi
if inside:
    checks['dml_ci_coverage'] = flag(
        round(pub_coef, 6),
        pass_cond=True,
        warn_cond=True,
        note=f'published={pub_coef} inside DML CI [{dml_lo:.4f}, {dml_hi:.4f}]'
    )
else:
    dist = min(abs(pub_coef - dml_lo), abs(pub_coef - dml_hi))
    within_1se = dist <= dml_se if dml_se else False
    within_2se = dist <= 2 * dml_se if dml_se else False
    ci_note = f'published={pub_coef} outside DML CI [{dml_lo:.4f}, {dml_hi:.4f}], dist={dist:.4f}'
    checks['dml_ci_coverage'] = flag(
        round(pub_coef, 6),
        pass_cond=False,
        warn_cond=within_2se,
        note=ci_note
    )
print(f'Check 3 (CI coverage): {checks["dml_ci_coverage"]["status"]} -- {checks["dml_ci_coverage"]["note"]}')

# ═══════════════════════════════════════════════════════════════
# Check 4: Nuisance model quality
# Use the primary spec's per-learner nuisance R2 values
# PASS if r2 > 0.3, WARN if > 0.1, FAIL if < 0.1
# ═══════════════════════════════════════════════════════════════
r2_out_vals = []
r2_trt_vals = []
individual_learners = ['Lasso', 'ElasticNet', 'DecisionTree', 'Boosting', 'Forest', 'NeuralNet']

for lname in individual_learners:
    ldata = primary_spec['learners'].get(lname)
    if ldata and 'nuisance' in ldata and ldata['nuisance']:
        r2_o = ldata['nuisance'].get('r2_outcome')
        r2_t = ldata['nuisance'].get('r2_treatment')
        if r2_o is not None:
            r2_out_vals.append(r2_o)
        if r2_t is not None:
            r2_trt_vals.append(r2_t)

r2_out = float(np.mean(r2_out_vals)) if r2_out_vals else 0.0
r2_trt = float(np.mean(r2_trt_vals)) if r2_trt_vals else 0.0
best_r2_out = max(r2_out_vals) if r2_out_vals else r2_out
best_r2_trt = max(r2_trt_vals) if r2_trt_vals else r2_trt

nuisance_note = (f'avg r2_outcome={r2_out:.4f} (best={best_r2_out:.4f}), '
                 f'avg r2_treatment={r2_trt:.4f} (best={best_r2_trt:.4f})')

checks['nuisance_quality'] = flag(
    {'r2_outcome': round(r2_out, 4), 'r2_treatment': round(r2_trt, 4),
     'best_r2_outcome': round(best_r2_out, 4), 'best_r2_treatment': round(best_r2_trt, 4)},
    pass_cond=(r2_out > 0.3 and r2_trt > 0.3),
    warn_cond=(r2_out > 0.1 or r2_trt > 0.1),
    note=nuisance_note
)
print(f'Check 4 (Nuisance R2): {checks["nuisance_quality"]["status"]} -- {checks["nuisance_quality"]["note"]}')

# ═══════════════════════════════════════════════════════════════
# Check 5: Sample size
# PASS if n >= 50, WARN if [30,50), FAIL if < 30
# ═══════════════════════════════════════════════════════════════
sample_sizes = [s['n_obs'] for s in dml_res['specifications']]
n_min = min(sample_sizes)
n_max = max(sample_sizes)
n_note = f'n_min={n_min}, n_max={n_max} across {len(sample_sizes)} specs'

checks['sample_size'] = flag(
    n_min,
    pass_cond=(n_min >= 50),
    warn_cond=(n_min >= 30),
    note=n_note
)
print(f'Check 5 (Sample size): {checks["sample_size"]["status"]} -- {checks["sample_size"]["note"]}')

# ═══════════════════════════════════════════════════════════════
# Check 6: HTE heterogeneity signal (skip if no hte_results)
# PASS if heterogeneity_detected == true
# WARN if all GATE CIs overlap (homogeneous)
# ═══════════════════════════════════════════════════════════════
if hte_res is not None:
    # Check gate section (may be nested under 'gate')
    gate_data = hte_res.get('gate', hte_res)
    het_detected = gate_data.get('heterogeneity_detected', False)
    groups = gate_data.get('groups', [])
    group_summary = ', '.join([f'{g["label"]}={g["coef"]:.3f} [{g["ci_lo"]:.3f},{g["ci_hi"]:.3f}]'
                               for g in groups])
    checks['hte_heterogeneity'] = flag(
        het_detected,
        pass_cond=het_detected,
        warn_cond=True,  # if not PASS, always WARN (never FAIL for HTE)
        note=f'heterogeneity_detected={het_detected}; GATE groups: {group_summary}'
    )
    print(f'Check 6 (HTE): {checks["hte_heterogeneity"]["status"]} -- heterogeneity_detected={het_detected}')
else:
    print('Check 6 (HTE): SKIPPED -- no hte_results.json')

# ═══════════════════════════════════════════════════════════════
# Check 7: Causal Forest ATE consistency (skip if no CF results)
# ═══════════════════════════════════════════════════════════════
if cf_res is not None:
    cf_ate = cf_res.get('ate', cf_res.get('ate_mean'))
    if cf_ate is not None:
        # DML+CF pipeline: compare against DML
        if dml_coef is not None:
            same_sign_cf = (cf_ate * dml_coef) > 0
            cf_rel_diff = abs(cf_ate - dml_coef) / abs(dml_coef) if dml_coef != 0 else float('inf')
            checks['cf_ate_consistency'] = flag(
                round(cf_ate, 6),
                pass_cond=(same_sign_cf and cf_rel_diff < 0.5),
                warn_cond=same_sign_cf,
                note=f'cf_ate={cf_ate:.4f} vs dml={dml_coef:.4f}, rel_diff={cf_rel_diff:.1%}'
            )
        else:
            # CF-only pipeline: compare against published
            same_sign_cf = (cf_ate * pub_coef) > 0
            cf_rel_diff = abs(cf_ate - pub_coef) / abs(pub_coef) if pub_coef != 0 else float('inf')
            checks['cf_ate_consistency'] = flag(
                round(cf_ate, 6),
                pass_cond=(same_sign_cf),
                warn_cond=(cf_rel_diff < 0.5),
                note=f'cf_ate={cf_ate:.4f} vs published={pub_coef}, rel_diff={cf_rel_diff:.1%}'
            )
        print(f'Check 7 (CF ATE): {checks["cf_ate_consistency"]["status"]} -- {checks["cf_ate_consistency"]["note"]}')
    else:
        print('Check 7 (CF ATE): SKIPPED -- missing ate in causal_forest_results')
else:
    print('Check 7 (CF ATE): SKIPPED -- no causal_forest_results.json')

# ═══════════════════════════════════════════════════════════════
# Check 8: Causal Forest CATE plausibility (skip if no CF results)
# ═══════════════════════════════════════════════════════════════
if cf_res is not None:
    pct_sig = cf_res.get('pct_significant')
    n_cf = cf_res.get('n_obs', 0)
    if pct_sig is not None:
        overfitting = pct_sig > 0.99 and n_cf < 200
        checks['cf_cate_plausibility'] = flag(
            round(pct_sig, 4),
            pass_cond=(0.10 < pct_sig < 0.95),
            warn_cond=(not overfitting),
            note=f'{pct_sig:.1%} of CATEs significant (n={n_cf})'
        )
        print(f'Check 8 (CF CATE): {checks["cf_cate_plausibility"]["status"]} -- {checks["cf_cate_plausibility"]["note"]}')
    else:
        print('Check 8 (CF CATE): SKIPPED -- no pct_significant in causal_forest_results')
else:
    print('Check 8 (CF CATE): SKIPPED -- no causal_forest_results.json')

# ═══════════════════════════════════════════════════════════════
# Check 9: ATE SE plausibility (skip if CF or HTE absent)
# ═══════════════════════════════════════════════════════════════
if cf_res is not None and hte_res is not None:
    ate_ci_lo = cf_res.get('ate_ci_lo')
    ate_ci_hi = cf_res.get('ate_ci_hi')
    gate_data_9 = hte_res.get('gate', hte_res)
    groups_9 = gate_data_9.get('groups', [])
    if ate_ci_lo is not None and ate_ci_hi is not None and groups_9:
        ate_ci_width = ate_ci_hi - ate_ci_lo
        gate_widths = [g['ci_hi'] - g['ci_lo'] for g in groups_9]
        mean_gate_width = float(np.mean(gate_widths))
        ratio = mean_gate_width / ate_ci_width if ate_ci_width > 0 else float('inf')
        checks['ate_se_plausibility'] = flag(
            round(ratio, 2),
            pass_cond=(ratio < 5),
            warn_cond=(ratio < 10),
            note=f'GATE/ATE CI width ratio={ratio:.2f} (ate_width={ate_ci_width:.4f}, mean_gate_width={mean_gate_width:.4f})'
        )
        print(f'Check 9 (ATE SE): {checks["ate_se_plausibility"]["status"]} -- {checks["ate_se_plausibility"]["note"]}')
    else:
        print('Check 9 (ATE SE): SKIPPED -- missing CI bounds')
else:
    print('Check 9 (ATE SE): SKIPPED -- need both causal_forest_results and hte_results')

# ═══════════════════════════════════════════════════════════════
# Check 10: Cross-fitting stability
# sd_across_reps vs median_se for the Best learner of the primary spec
# PASS if sd < 0.5 * median_se
# WARN if sd in [0.5, 1.0] * median_se
# FAIL if sd > median_se
# ═══════════════════════════════════════════════════════════════
# Best learner inherits from the best individual method -- get per_rep data from that method
best_outcome_method = primary_spec['learners']['Best'].get('best_outcome_method')
best_learner_data = primary_spec['learners'].get(best_outcome_method) if best_outcome_method else None

if best_learner_data and best_learner_data.get('per_rep_coefs') and best_learner_data.get('per_rep_ses'):
    per_rep_coefs = np.array(best_learner_data['per_rep_coefs'])
    per_rep_ses = np.array(best_learner_data['per_rep_ses'])
    sd_across_reps = float(np.std(per_rep_coefs, ddof=1))
    median_se = float(np.median(per_rep_ses))
    ratio_cf = sd_across_reps / median_se if median_se > 0 else float('inf')

    cf_note = (f'sd_across_reps={sd_across_reps:.4f}, median_se={median_se:.4f}, '
               f'ratio={ratio_cf:.3f} (learner={best_outcome_method})')

    checks['crossfit_stability'] = flag(
        round(ratio_cf, 4),
        pass_cond=(sd_across_reps < 0.5 * median_se),
        warn_cond=(sd_across_reps < 1.0 * median_se),
        note=cf_note
    )
    print(f'Check 10 (Cross-fit stability): {checks["crossfit_stability"]["status"]} -- {checks["crossfit_stability"]["note"]}')
else:
    print('Check 10 (Cross-fit stability): SKIPPED -- per_rep_coefs not available')

# ═══════════════════════════════════════════════════════════════
# Check 11: Learner sign agreement
# Count how many individual learners agree on sign for primary spec
# PASS if all agree, WARN if majority, FAIL if evenly split
# ═══════════════════════════════════════════════════════════════
learner_coefs = {}
for lname in individual_learners:
    ldata = primary_spec['learners'].get(lname)
    if ldata and 'coef' in ldata:
        learner_coefs[lname] = ldata['coef']

if len(learner_coefs) > 1:
    n_negative = sum(1 for c in learner_coefs.values() if c < 0)
    n_positive = sum(1 for c in learner_coefs.values() if c >= 0)
    n_learners = len(learner_coefs)
    majority_agree = max(n_negative, n_positive) > n_learners / 2
    all_agree = n_negative == n_learners or n_positive == n_learners
    dominant_sign = 'negative' if n_negative >= n_positive else 'positive'

    sign_detail = ', '.join([f'{ln}={lc:.4f}' for ln, lc in learner_coefs.items()])
    sign_note = (f'{n_negative}/{n_learners} negative, {n_positive}/{n_learners} positive '
                 f'(dominant={dominant_sign}); {sign_detail}')

    checks['learner_sign_agreement'] = flag(
        {'n_negative': n_negative, 'n_positive': n_positive, 'n_learners': n_learners},
        pass_cond=all_agree,
        warn_cond=majority_agree,
        note=sign_note
    )
    print(f'Check 11 (Learner sign agreement): {checks["learner_sign_agreement"]["status"]} -- {n_negative}/{n_learners} negative')
else:
    print('Check 11 (Learner sign agreement): SKIPPED -- only one learner')

# ═══════════════════════════════════════════════════════════════
# Check 12: Lasso variable selection
# PASS if n_nonzero > 0 for both outcome and treatment nuisance models
# WARN if n_nonzero == 0 for either model
# ═══════════════════════════════════════════════════════════════
lasso_data = primary_spec['learners'].get('Lasso')
lasso_diag = lasso_data.get('lasso_diagnostics') if lasso_data else None

if lasso_diag is not None:
    n_nonzero_outcome = lasso_diag.get('n_nonzero_outcome', None)
    n_nonzero_treatment = lasso_diag.get('n_nonzero_treatment', None)

    if n_nonzero_outcome is not None and n_nonzero_treatment is not None:
        both_nonzero = n_nonzero_outcome > 0 and n_nonzero_treatment > 0
        lasso_note = f'n_nonzero_outcome={n_nonzero_outcome}, n_nonzero_treatment={n_nonzero_treatment}'

        checks['lasso_variable_selection'] = flag(
            {'n_nonzero_outcome': n_nonzero_outcome, 'n_nonzero_treatment': n_nonzero_treatment},
            pass_cond=both_nonzero,
            warn_cond=True,  # if not PASS, always WARN (degenerate but not fatal)
            note=lasso_note
        )
        print(f'Check 12 (Lasso variable selection): {checks["lasso_variable_selection"]["status"]} -- {checks["lasso_variable_selection"]["note"]}')
    else:
        print('Check 12 (Lasso variable selection): SKIPPED -- lasso_diagnostics fields missing')
else:
    print('Check 12 (Lasso variable selection): SKIPPED -- lasso_diagnostics is null')

# ═══════════════════════════════════════════════════════════════
# Additional diagnostic: cross-specification sign consistency
# ═══════════════════════════════════════════════════════════════
print('\n-- Cross-specification sign consistency --')
sign_consistent = 0
sign_total = 0
for s in dml_res['specifications']:
    out_var = s['outcome']['variable'] if isinstance(s['outcome'], dict) else s['outcome']
    trt_var = s['treatment']['variable'] if isinstance(s['treatment'], dict) else s['treatment']
    ols_sign = np.sign(s['ols_coef'])
    best_coef = s['learners']['Best']['coef']
    dml_sign = np.sign(best_coef)
    match = ols_sign == dml_sign
    sign_consistent += int(match)
    sign_total += 1
    icon = 'OK' if match else 'XX'
    print(f'  [{icon}] {out_var} ~ {trt_var}: OLS={s["ols_coef"]:.4f}, DML(Best)={best_coef:.4f}')

print(f'  Sign agreement: {sign_consistent}/{sign_total}')

# ═══════════════════════════════════════════════════════════════
# Additional diagnostic: DML convergence check (NaN/inf)
# ═══════════════════════════════════════════════════════════════
print('\n-- DML convergence check --')
nan_count = 0
for s in dml_res['specifications']:
    for lname in individual_learners:
        ldata = s['learners'].get(lname)
        if ldata:
            for field in ['coef', 'se', 'pval']:
                v = ldata.get(field)
                if v is not None and (np.isnan(v) or np.isinf(v)):
                    nan_count += 1
                    out_var = s['outcome']['variable'] if isinstance(s['outcome'], dict) else s['outcome']
                    trt_var = s['treatment']['variable'] if isinstance(s['treatment'], dict) else s['treatment']
                    print(f'  WARNING: {out_var}~{trt_var} {lname}.{field} = {v}')
if nan_count == 0:
    n_learner_estimates = len(dml_res['specifications']) * len(individual_learners)
    print(f'  All {n_learner_estimates} learner estimates finite ({len(dml_res["specifications"])} specs x {len(individual_learners)} learners)')
else:
    print(f'  {nan_count} NaN/inf values detected')

# ═══════════════════════════════════════════════════════════════
# Summarise and write output
# ═══════════════════════════════════════════════════════════════
statuses = [v['status'] for v in checks.values()]
overall = 'FAIL' if 'FAIL' in statuses else ('WARN' if 'WARN' in statuses else 'PASS')
blocking = [k for k, v in checks.items() if v['status'] == 'FAIL']
warnings = [f"{k}: {v['note']}" for k, v in checks.items() if v['status'] == 'WARN']

diagnostics = {
    'overall': overall,
    'checks': checks,
    'blocking_issues': blocking,
    'warnings': warnings
}

DIAGNOSTICS_FLAGS.write_text(json.dumps(diagnostics, indent=2))

print(f'\n{"="*60}')
print(f'Overall: {overall}')
print(f'Blocking issues: {len(blocking)}')
for b in blocking:
    print(f'  [FAIL] {b}: {checks[b]["note"]}')
print(f'Warnings: {len(warnings)}')
for w in warnings:
    print(f'  [WARN] {w}')
print(f'Checks run: {len(checks)}')
print(f'{"="*60}')
print(f'\nWrote {DIAGNOSTICS_FLAGS}')
for k, v in checks.items():
    icon = {'PASS': 'PASS', 'WARN': 'WARN', 'FAIL': 'FAIL'}[v['status']]
    print(f'  [{icon}] {k}: {v["note"]}')

Check 1 (Replication): PASS -- 0.69% max deviation (4/4 within threshold); all signs correct
Check 2 (DML direction): PASS -- our=-0.2110 vs published=-0.178; same sign, rel_diff=18.5%
Check 3 (CI coverage): PASS -- published=-0.178 inside DML CI [-0.3663, -0.0556]
Check 4 (Nuisance R2): FAIL -- avg r2_outcome=-0.4265 (best=0.0508), avg r2_treatment=-0.1844 (best=0.0900)
Check 5 (Sample size): PASS -- n_min=50, n_max=61 across 12 specs
Check 6 (HTE): PASS -- heterogeneity_detected=True
Check 7 (CF ATE): SKIPPED -- no causal_forest_results.json
Check 8 (CF CATE): SKIPPED -- no causal_forest_results.json
Check 9 (ATE SE): SKIPPED -- need both causal_forest_results and hte_results
Check 10 (Cross-fit stability): PASS -- sd_across_reps=0.0159, median_se=0.0767, ratio=0.207 (learner=Forest)
Check 11 (Learner sign agreement): WARN -- 5/6 negative
Check 12 (Lasso variable selection): SKIPPED -- lasso_diagnostics is null

-- Cross-specification sign consistency --
  [OK] Investment2005 ~ statu